# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** among pages that are already visible in search, which ones should a content team review first to close a CTR gap — and can that prioritization be safely handed to a model instead of a transparent rule?

**Decision it supports:** review order for a content team's limited weekly capacity. Not which page to write, not whether to write at all — just which already-live page earns attention first.

In [1]:
print("Question: which visible pages should be reviewed first for a CTR gap, "
      "and does a model beat a transparent rule at ranking them?")
print("Decision supported: weekly content-review prioritization order.")

Question: which visible pages should be reviewed first for a CTR gap, and does a model beat a transparent rule at ranking them?
Decision supported: weekly content-review prioritization order.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Source: FlyRank starter CSV export ({len(df):,} content pieces, {df.shape[1]} columns)")
print("IDs are pseudonymous hashes (content_id, client_id) -- no client names, URLs, or raw queries.")
print("Window: trailing 90-day performance metrics (impressions, clicks, sessions); no absolute dates.")

pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
print(f"\nEligible (visible) pool: {len(pool):,} of {len(df):,} pages "
      f"({len(pool)/len(df):.0%}) -- impressions_90d >= 500 AND ranked in the top 20.")
print("Excluded from features: ctr/clicks_90d (the label's own source), trend_direction/trend_pct "
      "and the last_30d/prev_30d windows that build them (future/overlapping-window leakage), "
      "provider_used/model_used (per the data dictionary, not model features).")

Source: FlyRank starter CSV export (30,000 content pieces, 44 columns)
IDs are pseudonymous hashes (content_id, client_id) -- no client names, URLs, or raw queries.
Window: trailing 90-day performance metrics (impressions, clicks, sessions); no absolute dates.

Eligible (visible) pool: 12,023 of 30,000 pages (40%) -- impressions_90d >= 500 AND ranked in the top 20.
Excluded from features: ctr/clicks_90d (the label's own source), trend_direction/trend_pct and the last_30d/prev_30d windows that build them (future/overlapping-window leakage), provider_used/model_used (per the data dictionary, not model features).


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_opportunity` — CTR below the median CTR of other pages in the *same position tier* (top_3 / page_1 / striking), so a page is judged against fair peers, not the whole pool.

**Baseline rule:** `score = (tier_median_ctr − ctr) × impressions_90d` — an interpretable "estimated clicks/90d left on the table," with reason codes (`low_ctr_visible_page`, `leader_position_underperforming`, `high_volume_opportunity`, `weak_engagement_confirms`).

**Features:** 25 pre-outcome fields (content metadata + traffic history), with `has_<field>` flags added before filling real missingness — content_type drives which fields go missing, so a blind fillna would have quietly encoded content type as a feature.

**Validation design:** client-grouped 75/25 split (never split one client's pages across train and test), reused with the same seed across every modeling notebook.

**Leakage checks:** (1) deliberately reintroduced `ctr` as a feature and confirmed the harness catches it (AUC jumped 0.79 → 0.99); (2) confirmed no banned field (ctr, trend fields, provider/model, identifiers) is present in the final feature set; (3) audited the evaluation proxy itself and fixed a real bug — the engagement-corroboration check compared against a raw tier median that was exactly 0.0 for two tiers, making it fire almost never; fixed by comparing against the median of rows that actually measured engagement.

In [3]:
pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")
pool["is_opportunity"] = (pool.ctr < pool.tier_median_ctr).astype(int)
pool["ctr_gap"] = (pool["tier_median_ctr"] - pool["ctr"]).clip(lower=0)
pool["lost_clicks_90d"] = (pool["ctr_gap"] / 100) * pool["impressions_90d"]

measured_median = pool[pool.engagement_rate > 0].groupby("position_tier")["engagement_rate"].median()
pool["tier_median_engagement"] = pool["position_tier"].map(measured_median)
pool["confirmed_weak"] = (pool.engagement_rate > 0) & (pool.engagement_rate < pool["tier_median_engagement"])

numeric_feats = ["search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "days_since_last_update", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "ai_sessions_90d", "sessions_90d", "pageviews_90d", "users_90d",
    "engaged_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions"]
cat_feats = ["content_type", "main_intent", "competition_level",
             "word_count_tier", "char_count_tier", "age_tier", "freshness_tier"]
keep = ["content_id", "client_id", "is_opportunity", "lost_clicks_90d", "confirmed_weak"]
X = pool[numeric_feats + cat_feats + keep].copy()
for col in ["search_volume", "competition", "cpc", "word_count", "char_count"]:
    X[f"has_{col}"] = X[col].notna().astype(int)
    X[col] = X[col].fillna(X[col].median())
for col in numeric_feats:
    if X[col].isna().any() and f"has_{col}" not in X.columns:
        X[f"has_{col}"] = X[col].notna().astype(int)
        X[col] = X[col].fillna(X[col].median())
X = pd.get_dummies(X, columns=cat_feats, drop_first=True)
feature_cols = [c for c in X.columns if c not in keep]

banned = {"ctr", "clicks_90d", "trend_direction", "trend_pct", "avg_position", "position_tier",
          "impressions_90d", "impression_tier", "provider_used", "model_used",
          "content_id", "client_id"}
print(f"Features: {len(feature_cols)} | Leakage check: {'CLEAN' if not (banned & set(feature_cols)) else 'FAILED'}")

Features: 43 | Leakage check: CLEAN


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

SEED = 42
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(gss.split(X, groups=X["client_id"]))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]

scaler = StandardScaler()
lr = LogisticRegression(max_iter=2000, random_state=SEED).fit(scaler.fit_transform(Xtr[feature_cols]), Xtr["is_opportunity"])
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1).fit(Xtr[feature_cols], Xtr["is_opportunity"])

test = Xte.copy()
test["lr_prob"] = lr.predict_proba(scaler.transform(Xte[feature_cols]))[:, 1]
test["rf_prob"] = rf.predict_proba(Xte[feature_cols])[:, 1]

def precision_at_k(frame, col, k):
    return frame.sort_values(col, ascending=False)["confirmed_weak"].head(k).mean()

base_rate = test["confirmed_weak"].mean()
results = pd.DataFrame([
    {"k": k, "rule_baseline": precision_at_k(test, "lost_clicks_90d", k),
     "logistic_regression": precision_at_k(test, "lr_prob", k),
     "random_forest": precision_at_k(test, "rf_prob", k), "base_rate": base_rate}
    for k in [20, 50, 100, 200]
]).set_index("k").round(3)
print(f"Test set: {len(test)} pages, {test.client_id.nunique()} held-out clients, base rate={base_rate:.3f}\n")
results

Test set: 824 pages, 7 held-out clients, base rate=0.176



,rule_baseline,logistic_regression,random_forest,base_rate
k,,,,
20,0.250,0.00,0.000,0.176
50,0.160,0.02,0.000,0.176
100,0.150,0.01,0.000,0.176
200,0.085,0.01,0.005,0.176


**Headline result:** the transparent rule beats both models at every K. Both models rank *below* the base rate — their most-confident picks are exact-zero-engagement pages, which the evaluation proxy explicitly treats as "not measured," not "confirmed weak." The models are good at their own training label (Random Forest AUC 0.84) and bad at this yardstick — a mismatch between what was optimized and what was measured, not a broken model.

## 5. Limitations

*What this work cannot claim.*

- **Cross-sectional, not causal.** One 90-day snapshot. Nothing here shows that rewriting a title *causes* a CTR increase — only that some pages sit below their tier's typical CTR right now.
- **The evaluation proxy is not ground truth.** `confirmed_weak` (engagement below tier median) is a second signal correlated with CTR (ρ ≈ 0.3–0.5 by tier) but not independent proof a page is broken — precision@K against it measures corroboration, not accuracy against reality.
- **One starter export, not the full warehouse.** 30,000 rows across a handful of clients; FlyRank's production warehouse spans 341,701 pieces across 57 brands. Patterns here may not hold at that scale or across other verticals — this dataset has no vertical/industry field to check.
- **Selection and survivorship.** The eligible pool already filters to visible, high-traffic pages; anything below 500 impressions or ranked past #20 is invisible to this analysis, by design.
- **Anonymization limits.** Pseudonymous IDs mean no way to check whether a `client_id` cluster reflects one brand's editorial habits versus genuine signal — the client-grouped split protects against overfitting to this, but can't rule it out entirely.

In [5]:
print("Eligible pool covers", f"{len(pool)/len(df):.0%}", "of the starter export --",
      f"{len(df)-len(pool):,} pages are invisible to this analysis by the pool's own definition.")
print(f"confirmed_weak corroboration fires on {pool['confirmed_weak'].mean():.1%} of the eligible pool overall.")

Eligible pool covers 40% of the starter export -- 17,977 pages are invisible to this analysis by the pool's own definition.
confirmed_weak corroboration fires on 23.7% of the eligible pool overall.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

1. **Ship the rule-based queue, not the model** — it wins the honest comparison above, and it's the version a content editor can read and argue with.
2. **Review order:** start with `leader_position_underperforming` + `high_volume_opportunity` pages — well-ranked pages not converting that rank into clicks, where impression volume means a small CTR fix moves real click numbers.
3. **Treat `weak_engagement_confirms` as a bonus, not a gate** — only 15.6% of the queue carries it; the rest are still legitimate, just without the extra corroboration.
4. **Never automate the rewrite itself, or use this to judge writers** — the queue says where to look, never what to write, and CTR reflects more than writing quality.
5. **Re-check quarterly** — tier medians, the corroboration flag's fire rate, and content-type mix in the queue should all be monitored for drift as new 90-day data lands.

In [6]:
pool["reason_codes"] = None
hi_vol = pool["impressions_90d"].quantile(0.75)
def reason_codes(row):
    r = []
    if row.ctr < row.tier_median_ctr: r.append("low_ctr_visible_page")
    if row.position_tier in ("top_3", "page_1"): r.append("leader_position_underperforming")
    if row.impressions_90d >= hi_vol: r.append("high_volume_opportunity")
    if row.engagement_rate > 0 and row.engagement_rate < row.tier_median_engagement: r.append("weak_engagement_confirms")
    return "|".join(r) if r else "general_ctr_review"
pool["reason_codes"] = pool.apply(reason_codes, axis=1)
queue = pool[pool.ctr < pool.tier_median_ctr].sort_values("lost_clicks_90d", ascending=False)
print(f"Final queue: {len(queue):,} pages\n")
print(queue["reason_codes"].value_counts())

Final queue: 5,888 pages

reason_codes
low_ctr_visible_page|leader_position_underperforming                                                     2434
low_ctr_visible_page                                                                                     1816
low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity                              632
low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity|weak_engagement_confirms     464
low_ctr_visible_page|leader_position_underperforming|weak_engagement_confirms                             195
low_ctr_visible_page|weak_engagement_confirms                                                             184
low_ctr_visible_page|high_volume_opportunity                                                               89
low_ctr_visible_page|high_volume_opportunity|weak_engagement_confirms                                      74
Name: count, dtype: int64


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

img_dir = Path("../../docs/img")
img_dir.mkdir(parents=True, exist_ok=True)

# Chart 1: precision@K, rule vs models vs base rate
fig, ax = plt.subplots(figsize=(7, 4.2))
ks = results.index.tolist()
width = 0.2
x = np.arange(len(ks))
for i, col in enumerate(["rule_baseline", "logistic_regression", "random_forest"]):
    ax.bar(x + (i-1)*width, results[col], width, label=col.replace("_", " "))
ax.axhline(base_rate, color="gray", linestyle="--", linewidth=1, label=f"base rate ({base_rate:.2f})")
ax.set_xticks(x); ax.set_xticklabels([f"K={k}" for k in ks])
ax.set_ylabel("precision@K (engagement-confirmed)")
ax.set_title("Rule baseline vs. models, same held-out clients")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(img_dir / "precision_at_k.png", dpi=150)
plt.close(fig)

# Chart 2: reason code mix in the final queue
label_map = {
    "low_ctr_visible_page|leader_position_underperforming": "Underperforming rank",
    "low_ctr_visible_page": "Below-median CTR only",
    "low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity": "Underperforming rank + high volume",
    "low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity|weak_engagement_confirms": "All four signals",
    "low_ctr_visible_page|leader_position_underperforming|weak_engagement_confirms": "Underperforming rank + weak engagement",
    "low_ctr_visible_page|weak_engagement_confirms": "Below-median CTR + weak engagement",
}
fig, ax = plt.subplots(figsize=(7.6, 4.4))
counts = queue["reason_codes"].value_counts().head(6)
labels = [label_map.get(c, c) for c in counts.index]
ax.barh(labels[::-1], counts.values[::-1], color="#2F6F6B")
ax.set_xlabel("pages in the queue")
ax.set_title("Why pages are flagged (top reason-code combinations)")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.savefig(img_dir / "reason_codes.png", dpi=150)
plt.close(fig)

# Chart 3: CTR by position tier (grounds the whole tier-relative design)
fig, ax = plt.subplots(figsize=(7, 4.2))
tier_order = ["top_3", "page_1", "striking"]
medians = pool.groupby("position_tier")["ctr"].median().reindex(tier_order)
ax.bar(tier_order, medians.values, color="#55A868")
ax.set_ylabel("median CTR (%)")
ax.set_title("Median CTR by position tier (eligible pool)")
fig.tight_layout()
fig.savefig(img_dir / "ctr_by_tier.png", dpi=150)
plt.close(fig)

# Table for the paper
results.to_csv("../outputs/paper_precision_table.csv")

print("Saved: docs/img/precision_at_k.png, docs/img/reason_codes.png, docs/img/ctr_by_tier.png")
print("Saved: work/outputs/paper_precision_table.csv")

Saved: docs/img/precision_at_k.png, docs/img/reason_codes.png, docs/img/ctr_by_tier.png
Saved: work/outputs/paper_precision_table.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.